### Intialization

In [0]:
%sql
drop table workspace.silver.crm_products

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType,DataType,DateType


### # Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
print(f"total rows {df.count()}" )
df.display()

### silver transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

### Product Key Parsing

In [0]:
df = df.withColumn("cat_id",F.regexp_replace(F.substring(F.col("prd_key"),1,5),"-","_"))
df = df.withColumn("prd_key",F.substring(F.col("prd_key"),7,F.length(col("prd_key"))))

### Handling missing Cost 

In [0]:
df = df.withColumn("is_cost_known",F.col("prd_cost").isNotNull())
print(f"Products with unknown cost :{df.filter(F.col("prd_cost").isNull()).count()}")
print(f"Products with known cost :{df.filter(F.col("prd_cost").isNotNull()).count()}")

### Product Line Normalization

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "M","Mountain")
        .when(F.upper(F.col("prd_line"))== "R","Road")
        .when(F.upper(F.col("prd_line")) == "S","other sales")
        .when(F.upper(F.col("prd_line")) == "T","Touring")
        .otherwise("n/a")
    )
)

### Date casting

In [0]:
df = df.withColumn("prd_start_dt",F.col("prd_start_dt").cast(DateType()))

### Renaming columns

In [0]:
rename_map ={
    "prd_id":"product_id",
    "cat_id":"category_id",
    "prd_key":"product_number",
    "prd_nm":"product_name",
    "prd_cost":"product_cost",
    "prd_line":"product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt":"end_date"
}
for old_name,new_name in rename_map.items():
    df= df.withColumnRenamed(old_name,new_name)

### Sanity checks of dataframe

In [0]:
display(df.limit(10))
print("\n Product line distribution:")
df.groupBy("product_line").count().display()

print("\n Active vs historical products (end_date IS NULL =active):")
df.groupBy(F.col("end_date").isNull().alias("is_active")).count().display()


### Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")
print(f"Written {df.count()} rows to workspace.silver.crm_products")

In [0]:
%sql
select * from workspace.silver.crm_products;